# XAI — LRP attributions (CESM2-LE CNN)

Loads pre-trained CNN models and their TVT splits, then computes and saves LRP-z attributions for each split × seed.

> **Must run in a fresh kernel** — `src.xai.lrp` calls `tf.compat.v1.disable_eager_execution()` at import time.  
> Do **not** import or run this notebook in the same kernel session as `cnn_train.ipynb`.

**Prerequisites:** run `scripts/04_cesm2le_cnn_train.py` (or `cnn_train.ipynb`) to train the models first.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import netCDF4 as nc

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from configs import paths
from src.cnn.splits import load_tvt_split
from src.cnn.train  import load_model
from src.xai.lrp    import strip_sigmoid, compute_lrp_z, save_lrp

## Config
Match `N_SPLITS` and `N_SEEDS` to what was used during training.

In [ ]:
N_SPLITS   = 3     # number of TVT splits that were trained
N_SEEDS    = 3     # number of seeds per split
CHUNK_SIZE = 100   # samples per LRP batch — reduce if you hit memory errors

SKIP_EXISTING = True   # set False to recompute even if the output file exists

## Load lat / lon from CESM2-LE grid file

In [ ]:
ds_grid = nc.Dataset(paths.CESM2LE_GRID_FILE, 'r')
lat_1d  = np.array(ds_grid.variables['lat'][:])   # (nlat,)
lon_1d  = np.array(ds_grid.variables['lon'][:])   # (nlon,)
ds_grid.close()
print(f'Grid: lat {lat_1d.shape}, lon {lon_1d.shape}')

## Compute and save LRP for all splits × seeds

In [ ]:
for split_idx in range(N_SPLITS):
    print(f'\n{"─"*60}')
    print(f'Split {split_idx}')
    print(f'{"─"*60}')

    split_path = paths.tvt_split_path(split_idx)
    if not split_path.exists():
        raise FileNotFoundError(
            f'TVT split not found: {split_path}\n'
            f'Run scripts/03_cesm2le_tvt_splits.py first.'
        )
    split = load_tvt_split(split_path)
    x_tr  = split['sst_tr'][:, :, :, np.newaxis]   # (n_tr, nx, ny, 1)
    print(f'  Training data: {x_tr.shape}')

    for run_idx in range(N_SEEDS):
        out_path = paths.attribution_path(split_idx, run_idx)

        if SKIP_EXISTING and out_path.exists():
            print(f'  run {run_idx} — already exists, skipping.')
            continue

        model_p = paths.model_path(split_idx, run_idx)
        if not model_p.exists():
            print(f'  run {run_idx} — model not found ({model_p}), skipping.')
            continue

        print(f'  run {run_idx}: loading model ...', end=' ', flush=True)
        model        = load_model(paths.MODELS_DIR, split_idx, run_idx)
        model_logits = strip_sigmoid(model)
        print('done')

        print(f'  run {run_idx}: computing LRP (chunk_size={CHUNK_SIZE}) ...')
        attributions = compute_lrp_z(model_logits, x_tr, chunk_size=CHUNK_SIZE)
        # shape: (n_chunks, CHUNK_SIZE, nx, ny, 1)

        save_lrp(
            attributions, lat_1d, lon_1d,
            savepath=out_path,
            split_idx=split_idx,
            run_idx=run_idx,
        )

print('\nAll LRP attributions complete.')